[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/03.%20The%20Manual%20Reefer%20Monitoring%20Problem/P3-Tier-4_executed.ipynb)

---


# 3. The Manual Reefer Monitoring Problem

## Tier 4: Deep Q-Network Augmentation

This notebook formulates reefer inspections as a sequential decision problem and demonstrates a Deep Q-Network (DQN) agent. At each decision step, the agent selects a container-inspector pair, receives a reward, and observes the updated workload and inspection status. A small feedforward neural network approximates the value of each action.

### Goals, inputs, and outputs

- **Goal:** Learn a policy that selects inspection assignments with high cumulative reward while respecting inspector capacity.
- **Inputs:** Container attributes, inspector attributes, a fixed set of environmental conditions, and inspection-time and safety-risk data for each container-inspector pair.
- **Outputs:** Training metrics, an evaluation summary, a record of the policy's assignments, and visualizations of training progress.
- **Method:** Epsilon-greedy action selection, experience replay, a separate target network, and a hand-coded feedforward neural network trained by gradient descent.

### Model assumptions and scope

The example is a finite, simulated assignment problem. Container attributes and environmental conditions remain fixed during an episode. Inspector workloads and container inspection status change as actions are taken. Travel time is estimated from each inspector's initial location; inspectors do not move between inspections, and routes, equipment failures, changing temperatures, and new alarms are not simulated. Environmental conditions are included in the state representation but do not alter the transition or reward rules in this instance. The episode ends when every container has been inspected or no feasible assignment remains. Consequently, results describe this model and instance; they do not establish real-world performance or adaptation to changing conditions.

In [ ]:
from collections import deque, namedtuple
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import time

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## Formulation and solution sequence

A **state** is the information available to the agent before it chooses an assignment. An **action** is a container-inspector pair. A **reward** gives a numerical score to an assignment, combining time, safety risk, priority, urgency, alarm status, and workload utilization. The state records which containers have already been inspected; without that information, two situations with different remaining work could look identical.

The model is an episodic Markov decision process (MDP): given the current state and action, the next state and reward depend on the current modeled situation, not on an unrecorded history. Let $s_t$ denote the state at step $t$, $a_t$ the selected action, $r_t$ its reward, and $\gamma$ the discount factor. The action-value function $Q(s,a)$ estimates the expected discounted reward from taking action $a$ in state $s$ and then following the policy. Its Bellman optimality equation is

$$
Q^*(s,a) = \mathbb{E}\left[r_t + \gamma \max_{a' \in A(s_{t+1})} Q^*(s_{t+1},a') \mid s_t=s,\ a_t=a\right],
$$

where $A(s)$ is the set of feasible actions in state $s$. At a terminal transition, there is no future return, so the target is just $r_t$. The DQN approximates $Q(s,a)$ with a neural network. For a sampled transition, its training target is

$$
y_t = r_t + \gamma(1-d_t)\max_{a' \in A(s_{t+1})} Q_{\mathrm{target}}(s_{t+1},a'),
$$

where $d_t=1$ for a terminal transition and $d_t=0$ otherwise. If no next action is feasible, the transition is treated as having no future-value term. The network is trained to reduce the squared difference between its current estimate and this target for the selected action. The replay buffer stores past transitions so training can use a randomly sampled batch; the target network supplies a temporarily steadier estimate of future value.

The fixed action set is

$$
\mathcal{A}=\{(i,j): i\in\mathcal{C},\ j\in\mathcal{I}\},
$$

where $\mathcal{C}$ is the set of containers and $\mathcal{I}$ is the set of inspectors. The action mask selects the subset that is feasible in the current state: a pair is feasible only if its container is uninspected and the inspector has enough remaining capacity for the inspection and travel time. Choosing a feasible pair marks that container inspected and increases that inspector's workload. These rules define the modeled state transition; the mask also restricts greedy action selection and the maximum used in the Bellman target.

The state vector contains eight features per container, five per inspector, four environmental features, and two episode-progress features. Thus its dimension is $8n+5m+6$, where $n$ is the number of containers and $m$ is the number of inspectors. The network has $nm$ outputs, one per fixed container-inspector pair. The number of possible inspection subsets alone grows as $2^n$; workload and other state features add further possible states. This example therefore uses function approximation rather than exhaustive state enumeration.

The episode begins with all containers uninspected and all inspectors at zero workload. It terminates when all containers are inspected or there is no feasible assignment. The notebook follows this sequence: define the neural network and data structures; generate the instance; implement the environment and DQN; train and evaluate the agent; inspect its decisions and training curves; and study reward-weight sensitivity.

In [ ]:
class SimpleNeuralNetwork:
    """Feedforward network that approximates action values."""

    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.001):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate

        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, hidden_size) * 0.1
        self.b2 = np.zeros((1, hidden_size))
        self.W3 = np.random.randn(hidden_size, output_size) * 0.1
        self.b3 = np.zeros((1, output_size))

    def relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def forward(self, X):
        """Map a batch of state vectors to action-value estimates."""
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.relu(self.z2)
        self.z3 = np.dot(self.a2, self.W3) + self.b3
        return self.z3

    def backward(self, X, y, output):
        """Compute gradients for half the batch-mean sum of squared errors."""
        batch_size = X.shape[0]

        dZ3 = (output - y) / batch_size
        self.dW3 = np.dot(self.a2.T, dZ3)
        self.db3 = np.sum(dZ3, axis=0, keepdims=True)

        dA2 = np.dot(dZ3, self.W3.T)
        dZ2 = dA2 * self.relu_derivative(self.z2)
        self.dW2 = np.dot(self.a1.T, dZ2)
        self.db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.relu_derivative(self.z1)
        self.dW1 = np.dot(X.T, dZ1)
        self.db1 = np.sum(dZ1, axis=0, keepdims=True)

    def update_parameters(self):
        """Apply one gradient-descent update to every layer."""
        self.W1 -= self.learning_rate * self.dW1
        self.b1 -= self.learning_rate * self.db1
        self.W2 -= self.learning_rate * self.dW2
        self.b2 -= self.learning_rate * self.db2
        self.W3 -= self.learning_rate * self.dW3
        self.b3 -= self.learning_rate * self.db3

    def train(self, X, y, epochs=100):
        """Fit the network to supplied action-value targets."""
        for _ in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output)
            self.update_parameters()


Experience = namedtuple(
    "Experience",
    ["state", "action", "reward", "next_state", "done", "next_action_mask"],
)

In [ ]:
@dataclass
class Container:
    """A refrigerated container that requires inspection."""
    id: int
    priority: int
    stack_level: int
    location_x: float
    location_y: float
    max_inspection_interval: int
    time_since_last_inspection: float
    temperature: float
    alarm_status: bool


@dataclass
class Inspector:
    """An inspector with a workload limit and experience level."""
    id: int
    capacity_minutes: int
    experience_level: int
    current_location_x: float
    current_location_y: float
    current_workload: float = 0.0
    assigned_containers: Optional[List[int]] = None
    fatigue_level: float = 0.0

    def __post_init__(self):
        if self.assigned_containers is None:
            self.assigned_containers = []


@dataclass
class InspectionTime:
    """Time and risk data for one container-inspector pair."""
    container_id: int
    inspector_id: int
    time_minutes: int
    safety_risk_score: float
    travel_time: float


@dataclass
class EnvironmentState:
    """Structured record of the simulated environment."""
    container_status: List[Dict]
    inspector_status: List[Dict]
    environmental_conditions: Dict
    time_step: int
    total_inspections_completed: int

## What to examine in the results

Episode reward and loss curves are diagnostics, not proof that a globally optimal policy has been found. The evaluation reports the fraction of episodes that inspect every container, as well as reward and episode length. Because the environment instance is fixed, repeated evaluation episodes use the same underlying container and inspector data; they measure repeatability on that instance, not generalization to new conditions. Policy logs make individual assignments inspectable, while the sensitivity experiment shows how changing reward weights changes the training objective.

In [ ]:
def create_rl_environment():
    """Create the fixed simulated instance used throughout the notebook."""
    np.random.seed(42)
    random.seed(42)

    priority_distribution = [10] * 9 + [5] * 14 + [1] * 13
    np.random.shuffle(priority_distribution)

    stack_levels = [1] * 12 + [2] * 15 + [3] * 9
    np.random.shuffle(stack_levels)

    containers = []
    for index in range(36):
        priority = priority_distribution[index]
        containers.append(
            Container(
                id=index + 1,
                priority=priority,
                stack_level=stack_levels[index],
                location_x=np.random.uniform(0, 120),
                location_y=np.random.uniform(0, 120),
                max_inspection_interval={10: 2, 5: 3, 1: 4}[priority],
                time_since_last_inspection=np.random.uniform(0.5, 3.5),
                temperature=np.random.uniform(-18, 4),
                alarm_status=np.random.random() < 0.15,
            )
        )

    inspectors = [
        Inspector(1, 240, 3, 30, 30),
        Inspector(2, 240, 2, 90, 30),
        Inspector(3, 240, 2, 30, 90),
        Inspector(4, 240, 1, 90, 90),
    ]

    inspection_times = {}
    for container in containers:
        for inspector in inspectors:
            base_time = {1: 2, 2: 4, 3: 7}[container.stack_level]
            experience_factor = 1.0 - (inspector.experience_level - 1) * 0.15
            inspection_time = int(base_time * experience_factor)

            distance = np.sqrt(
                (container.location_x - inspector.current_location_x) ** 2
                + (container.location_y - inspector.current_location_y) ** 2
            )
            travel_time = distance / 10
            safety_risk = (
                container.stack_level
                * (4 - inspector.experience_level)
                * 0.5
            )
            inspection_times[(container.id, inspector.id)] = InspectionTime(
                container.id,
                inspector.id,
                inspection_time,
                safety_risk,
                travel_time,
            )

    environmental_conditions = {
        "weather_condition": np.random.choice(
            ["clear", "cloudy", "rain", "storm"]
        ),
        "visibility": np.random.uniform(0.5, 1.0),
        "equipment_status": np.random.choice(
            ["optimal", "degraded", "limited"]
        ),
        "traffic_congestion": np.random.uniform(0.0, 0.8),
    }
    return containers, inspectors, inspection_times, environmental_conditions


containers, inspectors, inspection_times, env_conditions = create_rl_environment()

priority_counts = {1: 0, 5: 0, 10: 0}
stack_counts = {1: 0, 2: 0, 3: 0}
alarm_count = 0
for container in containers:
    priority_counts[container.priority] += 1
    stack_counts[container.stack_level] += 1
    alarm_count += int(container.alarm_status)

print("=== REINFORCEMENT LEARNING INSTANCE ===")
print(f"Containers: {len(containers)}")
print(f"Priority distribution: {priority_counts}")
print(f"Stack-level distribution: {stack_counts}")
print(f"Containers with alarms: {alarm_count}")
print(f"Inspectors: {len(inspectors)}")
for inspector in inspectors:
    print(
        f"  Inspector {inspector.id}: experience={inspector.experience_level}, "
        f"capacity={inspector.capacity_minutes} minutes, "
        f"initial location=({inspector.current_location_x}, "
        f"{inspector.current_location_y})"
    )
print("Environmental conditions:")
for key, value in env_conditions.items():
    print(f"  {key}: {value}")

## Deep Q-Network implementation

The fixed output layer contains one Q-value for every container-inspector pair. The feasible-action mask prevents the policy from selecting an already inspected container or exceeding an inspector's capacity. The same mask is used when selecting the maximum next-state value in the Bellman target; otherwise, the network could assign value to an action the environment cannot execute.

For each sampled transition, the replay code copies the current network's action-value vector as a target vector and replaces the selected action's entry with its Bellman target. This makes the update focus on the chosen action while preserving the other current estimates for that training step. The reported loss is the mean squared error over the batch and action outputs; the gradient calculation corresponds to half the batch-mean sum of squared output errors.

The reward for an assignment is

$$
r = -0.4\frac{t}{10}
    + 0.3\frac{4-q}{4}
    + 0.2\frac{p}{10}
    + 0.1\frac{u}{h}
    + 0.5\,\mathbb{1}_{\mathrm{alarm}}
    - 0.2\max(0, v-0.9),
$$

where $t$ is inspection plus travel time, $q$ is the safety-risk score, $p$ is container priority, $u$ is time since the last inspection, $h$ is the maximum inspection interval, and $v$ is the inspector's workload utilization after assignment. The indicator is one when an alarm is active and zero otherwise. These terms express the chosen trade-offs in reward units; they are not physical measurements in a common unit. A larger reward is preferred by the agent.

In [ ]:
class ReeferMonitoringEnvironment:
    """Episodic assignment environment for reefer inspections."""

    def __init__(
        self, containers, inspectors, inspection_times, environmental_conditions
    ):
        self.containers = containers
        self.inspectors = inspectors
        self.inspection_times = inspection_times
        self.environmental_conditions = environmental_conditions
        self.valid_actions = [
            (container.id, inspector.id)
            for container in self.containers
            for inspector in self.inspectors
        ]
        self.state_size = self._calculate_state_size()
        self.action_size = len(self.valid_actions)
        self.reset()

    def reset(self):
        """Start a new episode with no assignments completed."""
        for inspector in self.inspectors:
            inspector.assigned_containers = []
            inspector.current_workload = 0.0
            inspector.fatigue_level = 0.0
        self.inspected_containers = set()
        self.time_step = 0
        return self._get_state()

    def _calculate_state_size(self):
        """Count every feature emitted by _get_state."""
        container_features = 8 * len(self.containers)
        inspector_features = 5 * len(self.inspectors)
        environmental_features = 4
        time_features = 2
        return (
            container_features
            + inspector_features
            + environmental_features
            + time_features
        )

    def _get_state(self):
        """Encode container, inspector, environment, and episode information."""
        state_vector = []

        for container in self.containers:
            state_vector.extend(
                [
                    container.priority / 10.0,
                    container.stack_level / 3.0,
                    container.time_since_last_inspection / 4.0,
                    (container.temperature + 20) / 25.0,
                    float(container.alarm_status),
                    float(container.id in self.inspected_containers),
                    container.location_x / 120.0,
                    container.location_y / 120.0,
                ]
            )

        for inspector in self.inspectors:
            state_vector.extend(
                [
                    inspector.experience_level / 3.0,
                    inspector.current_workload / inspector.capacity_minutes,
                    inspector.fatigue_level,
                    inspector.current_location_x / 120.0,
                    inspector.current_location_y / 120.0,
                ]
            )

        weather_encoding = {
            "clear": 0.0,
            "cloudy": 0.33,
            "rain": 0.67,
            "storm": 1.0,
        }
        equipment_encoding = {
            "optimal": 0.0,
            "degraded": 0.5,
            "limited": 1.0,
        }
        state_vector.extend(
            [
                weather_encoding[
                    self.environmental_conditions["weather_condition"]
                ],
                self.environmental_conditions["visibility"],
                equipment_encoding[
                    self.environmental_conditions["equipment_status"]
                ],
                self.environmental_conditions["traffic_congestion"],
                self.time_step / 100.0,
                len(self.inspected_containers) / len(self.containers),
            ]
        )
        state = np.asarray(state_vector, dtype=float)
        if state.size != self.state_size:
            raise ValueError("State vector length does not match state_size.")
        return state

    def get_action_mask(self):
        """Return a Boolean mask identifying currently feasible assignments."""
        mask = np.zeros(self.action_size, dtype=bool)
        container_by_id = {container.id: container for container in self.containers}
        inspector_by_id = {inspector.id: inspector for inspector in self.inspectors}

        for action_index, (container_id, inspector_id) in enumerate(
            self.valid_actions
        ):
            if container_id in self.inspected_containers:
                continue
            inspector = inspector_by_id[inspector_id]
            inspection = self.inspection_times[(container_id, inspector_id)]
            total_time = inspection.time_minutes + inspection.travel_time
            mask[action_index] = (
                inspector.current_workload + total_time
                <= inspector.capacity_minutes
            )
        return mask

    def step(self, action_index):
        """Execute one feasible assignment and return its transition."""
        if (
            not isinstance(action_index, (int, np.integer))
            or action_index < 0
            or action_index >= self.action_size
            or not self.get_action_mask()[action_index]
        ):
            return self._get_state(), -10.0, True, {"info": "Invalid action"}

        container_id, inspector_id = self.valid_actions[action_index]
        container = next(c for c in self.containers if c.id == container_id)
        inspector = next(i for i in self.inspectors if i.id == inspector_id)
        inspection = self.inspection_times[(container_id, inspector_id)]
        total_time = inspection.time_minutes + inspection.travel_time

        inspector.assigned_containers.append(container_id)
        inspector.current_workload += total_time
        inspector.fatigue_level = min(
            1.0,
            inspector.fatigue_level
            + total_time / inspector.capacity_minutes * 0.1,
        )
        self.inspected_containers.add(container_id)
        self.time_step += 1

        reward = self._calculate_reward(container, inspector, inspection)
        done = (
            len(self.inspected_containers) == len(self.containers)
            or not self.get_action_mask().any()
        )
        return self._get_state(), reward, done, {"info": "Inspection completed"}

    def _calculate_reward(self, container, inspector, inspection):
        """Score an assignment using the notebook's multi-term reward."""
        total_time = inspection.time_minutes + inspection.travel_time
        reward = -0.4 * total_time / 10.0
        reward += 0.3 * (4 - inspection.safety_risk_score) / 4.0
        reward += 0.2 * container.priority / 10.0
        reward += (
            0.1
            * container.time_since_last_inspection
            / container.max_inspection_interval
        )
        if container.alarm_status:
            reward += 0.5

        utilization = inspector.current_workload / inspector.capacity_minutes
        if utilization > 0.9:
            reward -= 0.2 * (utilization - 0.9)
        return reward


class DQNAgent:
    """DQN agent with epsilon-greedy exploration and replay memory."""

    def __init__(self, state_size, action_size, learning_rate=0.001):
        self.state_size = state_size
        self.action_size = action_size
        self.learning_rate = learning_rate
        self.q_network = SimpleNeuralNetwork(
            state_size, 64, action_size, learning_rate
        )
        self.target_network = SimpleNeuralNetwork(
            state_size, 64, action_size, learning_rate
        )
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 32
        self.update_target_every = 10
        self.training_history = {
            "episode_rewards": [],
            "episode_lengths": [],
            "epsilon_history": [],
            "loss_history": [],
        }
        self.update_target_network()

    def remember(self, state, action, reward, next_state, done, next_action_mask):
        """Store a transition and its feasible next-action set."""
        self.memory.append(
            Experience(
                state.copy(),
                action,
                reward,
                next_state.copy(),
                done,
                next_action_mask.copy(),
            )
        )

    def act(self, state, action_mask, training=True):
        """Choose a feasible action by epsilon-greedy selection."""
        feasible_actions = np.flatnonzero(action_mask)
        if feasible_actions.size == 0:
            return None
        if training and np.random.random() <= self.epsilon:
            return int(np.random.choice(feasible_actions))

        q_values = self.q_network.forward(state.reshape(1, -1))[0]
        feasible_values = q_values[feasible_actions]
        return int(feasible_actions[np.argmax(feasible_values)])

    def replay(self):
        """Update the Q-network from a random minibatch of stored transitions."""
        if len(self.memory) < self.batch_size:
            return

        batch = random.sample(self.memory, self.batch_size)
        states = np.asarray([item.state for item in batch])
        actions = np.asarray([item.action for item in batch], dtype=int)
        rewards = np.asarray([item.reward for item in batch], dtype=float)
        next_states = np.asarray([item.next_state for item in batch])
        dones = np.asarray([item.done for item in batch], dtype=bool)
        next_masks = np.asarray([item.next_action_mask for item in batch])

        current_q_values = self.q_network.forward(states)
        next_q_values = self.target_network.forward(next_states)
        target_q_values = current_q_values.copy()

        for index in range(self.batch_size):
            target = rewards[index]
            if not dones[index] and next_masks[index].any():
                feasible_values = next_q_values[index][next_masks[index]]
                target += self.gamma * np.max(feasible_values)
            target_q_values[index, actions[index]] = target

        self.q_network.train(states, target_q_values, epochs=1)
        predictions = self.q_network.forward(states)
        loss = np.mean((predictions - target_q_values) ** 2)
        self.training_history["loss_history"].append(float(loss))

    def update_target_network(self):
        """Copy the online network parameters into the target network."""
        for name in ("W1", "b1", "W2", "b2", "W3", "b3"):
            setattr(
                self.target_network,
                name,
                getattr(self.q_network, name).copy(),
            )

    def train(self, env, episodes):
        """Train for the requested number of episodes."""
        print(f"Starting DQN training for {episodes} episodes...")
        for episode in range(episodes):
            state = env.reset()
            total_reward = 0.0
            steps = 0
            done = False

            while not done and steps < 100:
                action_mask = env.get_action_mask()
                action = self.act(state, action_mask)
                if action is None:
                    break
                next_state, reward, done, _ = env.step(action)
                next_action_mask = env.get_action_mask()
                self.remember(
                    state,
                    action,
                    reward,
                    next_state,
                    done,
                    next_action_mask,
                )
                state = next_state
                total_reward += reward
                steps += 1

            self.replay()
            if episode % self.update_target_every == 0:
                self.update_target_network()
            self.epsilon = max(
                self.epsilon_min, self.epsilon * self.epsilon_decay
            )
            self.training_history["episode_rewards"].append(total_reward)
            self.training_history["episode_lengths"].append(steps)
            self.training_history["epsilon_history"].append(self.epsilon)

            if episode % 50 == 0:
                recent_rewards = self.training_history["episode_rewards"][-50:]
                print(
                    f"Episode {episode}: mean reward over recorded recent "
                    f"episodes={np.mean(recent_rewards):.2f}, "
                    f"epsilon={self.epsilon:.3f}, steps={steps}"
                )
        print("Training completed.")

    def evaluate(self, env, episodes=10):
        """Evaluate without exploration and count completed inspections."""
        total_rewards = []
        total_steps = []
        completed_episodes = 0

        for _ in range(episodes):
            state = env.reset()
            total_reward = 0.0
            steps = 0
            done = False

            while not done and steps < 100:
                action = self.act(state, env.get_action_mask(), training=False)
                if action is None:
                    break
                state, reward, done, _ = env.step(action)
                total_reward += reward
                steps += 1

            total_rewards.append(total_reward)
            total_steps.append(steps)
            if len(env.inspected_containers) == len(env.containers):
                completed_episodes += 1

        results = {
            "avg_reward": float(np.mean(total_rewards)),
            "avg_steps": float(np.mean(total_steps)),
            "success_rate": 100.0 * completed_episodes / episodes,
        }
        print(f"Evaluation over {episodes} episodes:")
        print(f"  Average reward: {results['avg_reward']:.2f}")
        print(f"  Average steps: {results['avg_steps']:.1f}")
        print(f"  Complete-inspection rate: {results['success_rate']:.1f}%")
        return results

In [ ]:
env = ReeferMonitoringEnvironment(
    containers, inspectors, inspection_times, env_conditions
)
agent = DQNAgent(env.state_size, env.action_size)

print("=== DEEP Q-NETWORK TRAINING ===")
print(f"State size: {env.state_size}")
print(f"Action size: {env.action_size}")
print(f"Initially feasible actions: {env.get_action_mask().sum()}")

start_time = time.time()
agent.train(env, episodes=500)
training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")

evaluation_results = agent.evaluate(env, episodes=20)

## Training progress visualization

The plots show the recorded episode rewards and lengths, the exploration rate, and the minibatch loss. A moving average reduces short-term visual variation but does not alter training. Reward and loss have different meanings: reward measures outcomes under the modeled objective, while loss measures the network's fit to sampled Bellman targets.

In [ ]:
def create_training_visualization(agent):
    """Plot the training metrics stored by the supplied agent."""
    history = agent.training_history
    episodes = np.arange(len(history["episode_rewards"]))
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("DQN Training Progress: Reefer Inspection Scheduling")

    axes[0, 0].plot(
        episodes, history["episode_rewards"], color="tab:blue", alpha=0.7
    )
    window_size = 20
    if len(episodes) >= window_size:
        moving_average = pd.Series(history["episode_rewards"]).rolling(
            window_size, min_periods=1
        ).mean()
        axes[0, 0].plot(
            episodes,
            moving_average,
            color="tab:red",
            label=f"{window_size}-episode moving average",
        )
        axes[0, 0].legend()
    axes[0, 0].set_xlabel("Episode")
    axes[0, 0].set_ylabel("Total reward (reward units)")
    axes[0, 0].set_title("Episode rewards")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        episodes, history["episode_lengths"], color="tab:green", alpha=0.8
    )
    axes[0, 1].set_xlabel("Episode")
    axes[0, 1].set_ylabel("Assignments per episode")
    axes[0, 1].set_title("Episode lengths")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        episodes, history["epsilon_history"], color="tab:purple"
    )
    axes[1, 0].set_xlabel("Episode")
    axes[1, 0].set_ylabel("Epsilon (exploration probability)")
    axes[1, 0].set_title("Epsilon-greedy exploration")
    axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].grid(True, alpha=0.3)

    if history["loss_history"]:
        loss_steps = np.arange(len(history["loss_history"]))
        axes[1, 1].plot(
            loss_steps, history["loss_history"], color="tab:orange", alpha=0.7
        )
        loss_window = 50
        if len(loss_steps) >= loss_window:
            loss_average = pd.Series(history["loss_history"]).rolling(
                loss_window, min_periods=1
            ).mean()
            axes[1, 1].plot(
                loss_steps,
                loss_average,
                color="tab:red",
                label=f"{loss_window}-update moving average",
            )
            axes[1, 1].legend()
    axes[1, 1].set_xlabel("Replay update")
    axes[1, 1].set_ylabel("Mean squared Bellman-target error")
    axes[1, 1].set_title("Training loss")
    axes[1, 1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


create_training_visualization(agent)

## Learned policy analysis

This inspection run records the greedy policy's assignments. The logged Q-value is the network's estimate for the chosen state-action pair, not a probability or a guaranteed future reward. The capacity and inspection-status mask restricts each decision to feasible assignments.

In [ ]:
def analyze_learned_policy(env, agent, containers, inspectors, inspection_times):
    """Run and summarize the greedy policy on one episode."""
    state = env.reset()
    done = False
    step = 0
    inspection_log = []
    container_by_id = {container.id: container for container in containers}
    inspector_by_id = {inspector.id: inspector for inspector in inspectors}

    while not done and step < 100:
        action_mask = env.get_action_mask()
        action = agent.act(state, action_mask, training=False)
        if action is None:
            break
        q_values = agent.q_network.forward(state.reshape(1, -1))[0]
        container_id, inspector_id = env.valid_actions[action]
        next_state, reward, done, _ = env.step(action)
        container = container_by_id[container_id]
        inspector = inspector_by_id[inspector_id]
        inspection = inspection_times[(container_id, inspector_id)]

        inspection_log.append(
            {
                "step": step,
                "container_id": container_id,
                "inspector_id": inspector_id,
                "priority": container.priority,
                "stack_level": container.stack_level,
                "alarm_status": container.alarm_status,
                "inspector_level": inspector.experience_level,
                "total_time": inspection.time_minutes + inspection.travel_time,
                "safety_risk": inspection.safety_risk_score,
                "reward": reward,
                "q_value": q_values[action],
            }
        )
        state = next_state
        step += 1

    result = pd.DataFrame(inspection_log)
    complete = len(env.inspected_containers) == len(env.containers)
    print("=== GREEDY POLICY EXECUTION ===")
    print(f"Assignments made: {step}")
    print(f"Containers inspected: {len(env.inspected_containers)}")
    print(f"All containers inspected: {'Yes' if complete else 'No'}")

    if not result.empty:
        print(f"Mean reward per assignment: {result['reward'].mean():.3f}")
        print(f"Mean selected Q-value: {result['q_value'].mean():.3f}")
        alarm_assignments = result[result["alarm_status"]]
        if not alarm_assignments.empty:
            print(
                "Mean assignment step for containers with alarms: "
                f"{alarm_assignments['step'].mean():.1f}"
            )
    return result


policy_analysis = analyze_learned_policy(
    env, agent, containers, inspectors, inspection_times
)

## Evaluation interpretation

The evaluation statistics summarize the trained DQN on the specified instance. Complete-inspection rate is the proportion of evaluation episodes in which every container is assigned. Average reward sums the modeled reward terms over an episode; it is meaningful only relative to the same reward definition and instance. These metrics are not a comparison with other algorithms and do not establish an optimality gap.

In [ ]:
def summarize_evaluation(evaluation_results, agent, env):
    """Report DQN evaluation metrics and basic model dimensions."""
    print("=== DQN EVALUATION SUMMARY ===")
    print(f"State features: {env.state_size}")
    print(f"Available assignment pairs: {env.action_size}")
    print(f"Mean evaluation reward: {evaluation_results['avg_reward']:.2f}")
    print(f"Mean assignments per episode: {evaluation_results['avg_steps']:.1f}")
    print(
        "Complete-inspection rate: "
        f"{evaluation_results['success_rate']:.1f}%"
    )
    print(f"Final exploration probability: {agent.epsilon:.3f}")


summarize_evaluation(evaluation_results, agent, env)

## Why formulate the task as sequential decision-making?

A complete assignment plan could be viewed as a single decision, but this example instead chooses assignments one at a time. That choice makes the current workload and remaining containers part of the evolving state, and lets the reward express the value assigned to each inspection decision. The DQN is useful here as an illustration of learned action-value approximation; it does not guarantee optimality, explain its decisions fully, or automatically adapt to conditions absent from its training environment.

The main modeling trade-off is explicit: reward weights encode priorities, while state features and transition rules determine what the agent can observe and learn. Any operational use would require validating those assumptions and the simulated dynamics against the intended terminal process.

## Reward-function sensitivity analysis

The sensitivity experiment keeps the DQN approach and instance structure, while changing the relative weights on time, safety, priority, and urgency. Alarm and high-utilization terms retain their specified coefficients. Each scenario uses a reproducible seed for network initialization, exploration, and replay sampling. Since changing a reward changes the learning objective, scores across scenarios should be interpreted alongside each scenario's weights rather than treated as measurements on a common operational scale.

In [ ]:
def reward_function_sensitivity(
    containers, inspectors, inspection_times, environmental_conditions
):
    """Train and evaluate DQN agents under several reward-weight settings."""
    reward_scenarios = [
        {
            "time_weight": 0.4,
            "safety_weight": 0.3,
            "priority_weight": 0.2,
            "urgency_weight": 0.1,
            "name": "Balanced reward",
        },
        {
            "time_weight": 0.6,
            "safety_weight": 0.2,
            "priority_weight": 0.1,
            "urgency_weight": 0.1,
            "name": "Efficiency focused",
        },
        {
            "time_weight": 0.2,
            "safety_weight": 0.5,
            "priority_weight": 0.2,
            "urgency_weight": 0.1,
            "name": "Safety focused",
        },
        {
            "time_weight": 0.2,
            "safety_weight": 0.2,
            "priority_weight": 0.5,
            "urgency_weight": 0.1,
            "name": "Priority focused",
        },
        {
            "time_weight": 0.25,
            "safety_weight": 0.25,
            "priority_weight": 0.25,
            "urgency_weight": 0.25,
            "name": "Equal weights",
        },
    ]

    class CustomEnvironment(ReeferMonitoringEnvironment):
        """Environment whose reward coefficients are supplied at construction."""

        def __init__(
            self,
            containers,
            inspectors,
            inspection_times,
            environmental_conditions,
            reward_weights,
        ):
            self.reward_weights = reward_weights
            super().__init__(
                containers, inspectors, inspection_times, environmental_conditions
            )

        def _calculate_reward(self, container, inspector, inspection):
            total_time = inspection.time_minutes + inspection.travel_time
            weights = self.reward_weights
            reward = -weights["time_weight"] * total_time / 10.0
            reward += (
                weights["safety_weight"]
                * (4 - inspection.safety_risk_score)
                / 4.0
            )
            reward += weights["priority_weight"] * container.priority / 10.0
            reward += (
                weights["urgency_weight"]
                * container.time_since_last_inspection
                / container.max_inspection_interval
            )
            if container.alarm_status:
                reward += 0.5
            utilization = inspector.current_workload / inspector.capacity_minutes
            if utilization > 0.9:
                reward -= 0.2 * (utilization - 0.9)
            return reward

    results = []
    for scenario_index, scenario in enumerate(reward_scenarios):
        np.random.seed(42 + scenario_index)
        random.seed(42 + scenario_index)
        print(f"Training scenario: {scenario['name']}")
        custom_env = CustomEnvironment(
            containers,
            inspectors,
            inspection_times,
            environmental_conditions,
            scenario,
        )
        custom_agent = DQNAgent(custom_env.state_size, custom_env.action_size)
        start_time = time.time()
        custom_agent.train(custom_env, episodes=200)
        elapsed_time = time.time() - start_time
        eval_results = custom_agent.evaluate(custom_env, episodes=10)
        results.append(
            {
                "scenario": scenario["name"],
                "avg_reward": eval_results["avg_reward"],
                "success_rate": eval_results["success_rate"],
                "training_time_seconds": elapsed_time,
                "final_epsilon": custom_agent.epsilon,
            }
        )

    results_df = pd.DataFrame(results)
    print("\nReward-weight sensitivity results:")
    print(results_df.round(3).to_string(index=False))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Reward-Weight Sensitivity Analysis")
    metrics = [
        ("avg_reward", "Mean episode reward", "tab:blue"),
        ("success_rate", "Complete-inspection rate (%)", "tab:red"),
        ("training_time_seconds", "Training time (seconds)", "tab:green"),
        ("final_epsilon", "Final exploration probability", "tab:orange"),
    ]
    for axis, (column, ylabel, color) in zip(axes.flat, metrics):
        axis.bar(results_df["scenario"], results_df[column], color=color, alpha=0.75)
        axis.set_ylabel(ylabel)
        axis.set_title(ylabel)
        axis.tick_params(axis="x", rotation=35)
        axis.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()
    return results_df


reward_results = reward_function_sensitivity(
    containers, inspectors, inspection_times, env_conditions
)

## Summary

This notebook represents manual reefer monitoring as a finite sequential assignment problem and applies a Deep Q-Network to approximate action values. The implementation makes the main components explicit:

- **State:** normalized container attributes, inspection status, inspector workload and attributes, environmental conditions, and episode progress.
- **Action:** one container-inspector pair from a fixed output set, filtered by current inspection status and capacity.
- **Reward:** a weighted combination of time, safety risk, priority, urgency, alarm status, and workload utilization.
- **Learning:** epsilon-greedy action selection, a replay buffer, Bellman targets from a target network, and gradient descent in a hand-coded neural network.
- **Evaluation:** reward, episode length, completion rate, assignment logs, and training curves for the simulated instance.

The results should be interpreted within the model's scope. In particular, a falling loss does not prove policy optimality, a high completion rate on one fixed instance does not demonstrate generalization, and reward weights encode modeling choices rather than operational facts. The sensitivity analysis makes the dependence on those choices visible. Further work within this modeling approach could represent changing conditions or additional operational constraints, but would require defining their state features, transitions, feasibility rules, and rewards before training.